<a href="https://colab.research.google.com/github/cojocarucosmin/AICourseDev/blob/main/RAG_multiformat_llamaindex_Gradio_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Install libraries and download data

In [62]:
!pip install -q llama-index llama-index-llms-openai llama-index-readers-file openai gradio python-docx python-pptx pandas PyMuPDF docx2txt


In [ ]:
!mkdir -p 'data/'

# Download sample data
!wget 'https://raw.githubusercontent.com/run-llama/llama_index/main/docs/docs/examples/data/paul_graham/paul_graham_essay.txt' -O 'data/paul_graham_essay.txt'

In [59]:
import os
import gradio as gr
from pathlib import Path
from llama_index.llms.openai import OpenAI
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.readers.file.docs    import DocxReader, PDFReader
from llama_index.readers.file.slides  import PptxReader
from llama_index.readers.file.tabular import PandasExcelReader

In [60]:
# only for Google Colab; please comment Kaggle part in this case
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

## Chatbot with internal knowledge

For each chat interaction:
- first generate a standalone question from conversation context and last message, then
- query the query engine with the condensed question for a response.




In [76]:
input_dir = Path("./data/")

llm = OpenAI(model="gpt-4o", temperature=0)

# read all uploaded documents in the folder, irrespective of their format (e.g. pdf, xlsx, docx)
documents = SimpleDirectoryReader("./data").load_data()
index = VectorStoreIndex.from_documents(documents)

In [79]:
# show indexed content
shown = set()
for d in documents:
    f = d.metadata.get("file_path", d.id_)
    if f in shown:
        continue          # skip pages we've already shown
    shown.add(f)
    print(f, "\n", d.text[:100].replace("\n", " "), "\n" + "-"*20)

/content/data/Conditii contractuale Premium Care Abroad 2024.pdf 
 Condițiile generale ale contractului  de asigurare de sănătate   Premium Care Abroad  (Tratament Bol 
--------------------
/content/data/Gen_AI_Custom.docx 
 AI Generativ & Automatizare  Obiectiv: Oferirea unei perspective complete și aplicate asupra capabil 
--------------------
/content/data/Williams_Book_Store_Receipt.xlsx 
 Store Name: WILLIAMS' BOOK STORE, Established: 1908, Store Address: 708 South Pacific Avenue, San Pe 
--------------------
/content/data/paul_graham_essay.txt 
   What I Worked On  February 2021  Before college the two main things I worked on, outside of school 
--------------------


In [65]:
from llama_index.core.memory import ChatMemoryBuffer
from llama_index.core.prompts import PromptTemplate

# Create memory buffer
memory = ChatMemoryBuffer.from_defaults(token_limit=12000)

# 3) Define your prompt (including chat history + context)
context_prompt = PromptTemplate(
    "You are a chatbot that answers the user’s questions from provided documents.\n\n"
    "Chat history:\n"
    "{chat_history_str}\n\n"
    "Relevant documents:\n"
    "{context_str}\n\n"
    "Instruction: Use the chat history above or the documents to inform your answer."
)

# 4) Build the chat engine with the temperature-tuned LLM
chat_engine = index.as_chat_engine(
    chat_mode="condense_plus_context",
    memory=memory,
    llm=llm,
    context_prompt=context_prompt,
    verbose=False,
)

### Chat with your data

In [66]:
response = chat_engine.chat("Give me details about Williams Book Store?")
response.response

"Williams' Book Store was established in 1908 and is located at 708 South Pacific Avenue, San Pedro, California, 90731. The store's phone number is 832-3631."

In [67]:
response = chat_engine.chat("Who is Paul Graham?")
response.response

'Paul Graham is an essayist, programmer, and entrepreneur known for his work in the tech industry. He co-founded Viaweb, one of the first web-based applications, which was later acquired by Yahoo. He is also a co-founder of Y Combinator, a startup accelerator that has funded numerous successful startups. Graham is recognized for his essays on various topics, including technology, startups, and programming, and he has published a collection of essays titled "Hackers & Painters."'

In [68]:
response = chat_engine.chat("What do you know about Generative AI course?")
response.response

'The Generative AI course aims to provide a comprehensive and applied perspective on the capabilities of generative AI for processing text, documents, and internet searches. The course focuses on conceptual understanding, practical demonstrations, and relevant case studies. Participants will learn the fundamentals of generative AI and prompt engineering, explore integration with automation processes, and work hands-on with real-world scenarios such as data search and interpretation from competing platforms, information extraction from internal documents, code/part identification from logical diagrams, and using AI agents for research and data integration. The course offers both theoretical knowledge and practical skills applicable to business projects.'

### Use with Gradio

In [69]:
def predict(message, history):
    response = chat_engine.chat(message)
    return response.response

chat_ui = gr.ChatInterface(
    fn=predict,
    type="messages",
    title="AI Chatbot with Custom Knowledge",
    description="Knowledge retrieval ChatBot"
)

chat_ui.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://749a8083dd817578a9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### Reset chat engine

In [70]:
chat_engine.reset()

## Chat Engine - ReAct Agent Mode

ReAct is an agent based chat mode built on top of a query engine over your data.

For each chat interaction, the agent enter a ReAct loop:

- first decide whether to use the query engine tool and come up with appropriate input
- (optional) use the query engine tool and observe its output
- decide whether to repeat or give final response

In [72]:
from llama_index.core.agent import ReActAgent

# 1. Create a Query Engine from your index
query_engine = index.as_query_engine()

# 2. Wrap the Query Engine as a Tool
# This is crucial for the ReAct agent to "know" how to interact with your data.
query_engine_tool = QueryEngineTool(
    query_engine=query_engine,
    metadata=ToolMetadata(
        name="VectorStoreQueryEngine", # Give a descriptive name for your tool
        description=(
            "Useful for answering questions about the documents in the vector store. "
            "Use this tool to find specific information or answer general questions "
            "based on the provided data."
        ),
    ),
)

# 3. Initialize the ReActAgent with the tool(s) and your LLM
# The ReActAgent directly implements the ReAct loop.
# It takes a list of tools it can use.
chat_engine = ReActAgent.from_tools(
    tools=[query_engine_tool],
    llm=llm,
    verbose=True,
)

# 4. Interact with the chat engine
print("Chat engine initialized. Type your questions.")

Chat engine initialized. Type your questions.


/usr/local/lib/python3.11/dist-packages/llama_index/core/agent/react/base.py:154: DeprecationWarning: Call to deprecated class ReActAgent. (ReActAgent has been rewritten and replaced by llama_index.core.agent.workflow.ReActAgent.

This implementation will be removed in a v0.13.0 and the new implementation will be promoted to the `from llama_index.core.agent import ReActAgent` path.

See the docs for more information: https://docs.llamaindex.ai/en/stable/understanding/agent/)
  return cls(
/usr/local/lib/python3.11/dist-packages/deprecated/classic.py:184: DeprecationWarning: Call to deprecated class AgentRunner. (AgentRunner has been deprecated and is not maintained.

This implementation will be removed in a v0.13.0.

See the docs for more information on updated agent usage: https://docs.llamaindex.ai/en/stable/understanding/agent/)
  return old_new1(cls, *args, **kwargs)


In [73]:
response = chat_engine.chat("Use the tool to answer what Graham do in the summer of 1995?")
print(response)

> Running step 9ad369fa-4923-48f0-9b8e-e541e70803b1. Step input: Use the tool to answer what Graham do in the summer of 1995?
Thought: The current language of the user is English. I need to use a tool to help me answer the question.
Action: VectorStoreQueryEngine
Action Input: {'input': 'Graham in the summer of 1995'}
Observation: Graham in the summer of 1995 was focused on running Viaweb and experiencing a period of high productivity.
> Running step 27ef8c39-5272-4868-a18e-0fdb5c246cc8. Step input: None
Thought: I can answer without using any more tools. I'll use the user's language to answer.
Answer: In the summer of 1995, Graham was focused on running Viaweb and experiencing a period of high productivity.
In the summer of 1995, Graham was focused on running Viaweb and experiencing a period of high productivity.


In [74]:
response = chat_engine.chat("Is there any mention about 1995?")
print(response)

> Running step 3ef6a155-f735-4165-b75d-941e7802ad9e. Step input: Is there any mention about 1995?
Thought: The current language of the user is English. I need to use a tool to help me answer the question.
Action: VectorStoreQueryEngine
Action Input: {'input': '1995'}
Observation: 1995 was not specifically mentioned in the provided context.
> Running step 5efad688-dc68-4f49-af4b-a58f0ef8d699. Step input: None
Thought: I cannot answer the question with the provided tools.
Answer: There is no specific mention of the year 1995 in the provided context.
There is no specific mention of the year 1995 in the provided context.


In [75]:
response = chat_engine.chat("Who is Klaus Iohannis?")
print(response)

> Running step cc84aa09-38f5-49b0-a1ed-1600f829ce0e. Step input: Who is Klaus Iohannis?
Thought: The current language of the user is English. I need to use a tool to help me answer the question.
Action: VectorStoreQueryEngine
Action Input: {'input': 'Klaus Iohannis'}
Observation: Klaus Iohannis is a Romanian politician who has been serving as the President of Romania since December 21, 2014.
> Running step 1b7475f9-d134-4162-a830-7f86d43a73d1. Step input: None
Thought: I can answer without using any more tools. I'll use the user's language to answer.
Answer: Klaus Iohannis is a Romanian politician who has been serving as the President of Romania since December 21, 2014.
Klaus Iohannis is a Romanian politician who has been serving as the President of Romania since December 21, 2014.
